In [0]:
-- DESCRIBE DETAILS OF TABLE (FORMAT, PARTITION COLUMNS, SIZE, LOCATION )
DESCRIBE DETAIL susep.bronze_ses_uf2;

In [0]:
-- SHOW PARTITIONS (COLUMNS AND VALUES)
SHOW PARTITIONS susep.bronze_ses_uf2;

In [0]:
-- GRUPOS DE RAMOS
SELECT
    gracodigo,
    granome
  FROM
    susep.bronze_ses_gruposramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_gruposramos
      )
    ORDER BY 1

In [0]:
 -- RAMOS
  SELECT
    coramo,
    noramo
  FROM
    susep.bronze_ses_ramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_ramos
      )

In [0]:
-- EXPLORATION RAMOS E GRUPOS
WITH cte_depara_ramos_grupo AS (
  SELECT DISTINCT
    ramos,
    gracodigo
  from
    susep.bronze_ses_uf2
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_uf2
      )
),
cte_ramos_grupo_agg as (
  SELECT
    ramos,
    gracodigo,
    count(*) over (partition by ramos) as total_grupo_ramos
  FROM
    cte_depara_ramos_grupo
),
cte_ramos AS (
  SELECT
    coramo,
    noramo
  FROM
    susep.bronze_ses_ramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_ramos
      )
),
cte_grupos_ramos AS (
  SELECT
    gracodigo,
    granome
  FROM
    susep.bronze_ses_gruposramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_gruposramos
      )
)
SELECT
  cte_ramos.*,
  trim(regexp_replace(cte_ramos.noramo, '^[0-9]+\\s*-\\s*', '')) as nome_ramo,
  cte_grupos_ramos.*,
   trim(regexp_replace(cte_grupos_ramos.granome, '^[0-9]+\\s*-\\s*', '')) as nome_grupo_ramo
FROM
  cte_ramos_grupo_agg
    left join cte_ramos
      on cte_ramos_grupo_agg.ramos = cte_ramos.coramo
    left join cte_grupos_ramos
      on cte_ramos_grupo_agg.gracodigo = cte_grupos_ramos.gracodigo
ORDER BY nome_ramo


In [0]:
-- RAMOS NOT EXISTS IN TABLE UF
SELECT distinct
    coramo 
FROM susep.bronze_ses_seguros
where coramo not in (
  SELECT DISTINCT
    ramos
  from
    susep.bronze_ses_uf2
)

In [0]:
-- RAMOS NOT EXISTS IN TABLE RAMOS
SELECT 
    coramo,
    min(damesano) as min_damesano,
    max(damesano) as max_damesano 
FROM susep.bronze_ses_seguros
where
  ingestion_batch_id
    = (
      select
        max(ingestion_batch_id)
      from
        susep.bronze_ses_seguros
    )
  and coramo not in (
    SELECT DISTINCT
      coramo
    from
      susep.bronze_ses_ramos
    where
      ingestion_batch_id
        = (
          select
            max(ingestion_batch_id)
          from
            susep.bronze_ses_ramos
        )
  )
  group by coramo

In [0]:
-- RAMOS DISTINTCS
SELECT distinct
    coramo 
FROM susep.bronze_ses_seguros
where
  ingestion_batch_id
    = (
      select
        max(ingestion_batch_id)
      from
        susep.bronze_ses_seguros
    )

In [0]:
-- EMPRESAS

WITH base_limpa AS (
    SELECT DISTINCT
        coenti,
        TRIM(noenti) AS noenti,
        cogrupo,
        TRIM(nogrupo) AS nogrupo,
        UPPER(CONCAT(COALESCE(TRIM(nogrupo), ''), ' | ', COALESCE(TRIM(noenti), ''))) AS search_string
    FROM susep_bronze.ses_grupos_economicos
),

base_full AS
(
SELECT 
    *,
    CASE 
        -- 1. Grandes Conglomerados Bancários (Agrupando subsidiárias e marcas adquiridas)
        WHEN search_string LIKE '%BRADESCO%' OR search_string LIKE '%BCN%' OR search_string LIKE '%FINASA%' OR search_string LIKE '%BAMÉRCIO%' OR search_string LIKE '%BANERJ%' THEN 'BRADESCO SEGUROS'
        WHEN search_string LIKE '%ITAÚ%' OR search_string LIKE '%ITAU%' OR search_string LIKE '%UNIBANCO%' OR search_string LIKE '%BEMGE%' OR search_string LIKE '%BANCRED%' THEN 'ITAÚ UNIBANCO'
        WHEN search_string LIKE '%SANTANDER%' OR search_string LIKE '%REAL%' OR search_string LIKE '%HSBC%' OR search_string LIKE '%BAMERINDUS%' OR search_string LIKE '%AMÉRICA DO SUL%' OR search_string LIKE '%BBV%' THEN 'SANTANDER'
        WHEN search_string LIKE '%CAIXA%' THEN 'CAIXA SEGURIDADE'
        WHEN search_string LIKE '%SAFRA%' THEN 'SAFRA'
        WHEN search_string LIKE '%BANCO PACTUAL%' OR search_string LIKE '%PACTUAL%' THEN 'BTG PACTUAL'
        
        -- 2. Grandes Seguradoras Nacionais e Internacionais
        WHEN search_string LIKE '%PORTO SEGURO%' THEN 'PORTO SEGURO'
        WHEN search_string LIKE '%SUL AMERICA%' OR search_string LIKE '%SULAMÉRICA%' OR search_string LIKE '%SULINA%' THEN 'SULAMÉRICA'
        WHEN search_string LIKE '%MAPFRE%' OR search_string LIKE '%BBMAPFRE%' THEN 'MAPFRE'
        WHEN search_string LIKE '%ALLIANZ%' OR search_string LIKE '%AGF%' THEN 'ALLIANZ'
        WHEN search_string LIKE '%ZURICH%' OR search_string LIKE '%MINAS-BRASIL%' THEN 'ZURICH'
        WHEN search_string LIKE '%CHUBB%' OR search_string LIKE '%ACE%' OR search_string LIKE '%NOVA YORK%' THEN 'CHUBB'
        WHEN search_string LIKE '%LIBERTY%' OR search_string LIKE '%INDIANA%' OR search_string LIKE '%PAULISTA%' THEN 'LIBERTY'
        WHEN search_string LIKE '%TOKIO MARINE%' THEN 'TOKIO MARINE'
        
        -- 3. Grupos de Saúde / Operadoras Grandes
        WHEN search_string LIKE '%AMIL%' OR search_string LIKE '%BLUE LIFE%' OR search_string LIKE '%UNITED%' THEN 'AMIL'
        WHEN search_string LIKE '%UNIMED%' THEN 'UNIMED'
        WHEN search_string LIKE '%NOTRE DAME%' OR search_string LIKE '%GOLDEN CROSS INTERMÉDICA%' THEN 'NOTRE DAME INTERMÉDICA'
        
        -- 4. Grandes Especializadas e Tradicionais Consolidadas
        WHEN search_string LIKE '%HDI%' OR search_string LIKE '%HANNOVER%' OR search_string LIKE '%TALANX%' THEN 'HDI SEGUROS'
        WHEN search_string LIKE '%POTTENCIAL%' THEN 'POTTENCIAL'
        WHEN search_string LIKE '%EULER HERMES%' THEN 'EULER HERMES'
        WHEN search_string LIKE '%YASUDA%' OR search_string LIKE '%MARÍTIMA%' THEN 'SOMPO SEGUROS'
        WHEN search_string LIKE '%GENERALI%' OR search_string LIKE '%GENERALLI%' THEN 'GENERALI'
        WHEN search_string LIKE '%METROPOLITAN%' OR search_string LIKE '%METLIFE%' THEN 'METLIFE'
        WHEN search_string LIKE '%CARDIF%' OR search_string LIKE '%LUIZASEG%' THEN 'BNP PARIBAS CARDIF'
        WHEN search_string LIKE '%ICATU%' THEN 'ICATU'
        WHEN search_string LIKE '%SABEMI%' THEN 'SABEMI'
        WHEN search_string LIKE '%GBOEX%' OR search_string LIKE '%CONFIANCA%' THEN 'GBOEX-CONFIANÇA'

        -- O Funil Estrito: Tudo o que for pulverizado, independente antigo ou genérico vira 'OUTROS'
        ELSE 'OUTROS'
    END AS nome_empresa_consolidada

FROM base_limpa)

SELECT DISTINCT 
    *
FROM base_full
;


In [0]:
-- ============================================================
-- 4. LOAD SILVER TABLE FROM BRONZE
-- ============================================================
-- This step performs the joins, datatype treatment, code normalization,
-- date derivation, market classification and Silver hash generation.
  INSERT OVERWRITE TABLE susep_silver.tb_seguro_mensal_susep (
  nr_ano_mes_referencia,
  nr_ano_referencia,
  nr_mes_referencia,
  dt_referencia,
  cd_entidade_susep,
  nm_entidade_susep,
  cd_grupo_empresa_susep_original,
  cd_grupo_empresa_susep_mapeado,
  nm_grupo_empresa_susep_mapeado,
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_susep,
  nm_ramo_susep,
  cd_ramo_tratado,
  nm_ramo_tratado,
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude,
  ds_criterio_classificacao_mercado,
  vl_premio_direto,
  vl_premio_de_seguros,
  vl_premio_retido,
  vl_premio_ganho,
  vl_sinistro_direto,
  vl_sinistro_retido,
  vl_despesa_comercializacao,
  vl_premio_emitido,
  vl_premio_emitido_capitalizacao,
  vl_despesa_resseguro,
  vl_sinistro_ocorrido,
  vl_receita_resseguro,
  vl_sinistro_ocorrido_capitalizacao,
  vl_recuperacao_sinistro_ocorrido_capitalizacao,
  vl_rvne,
  vl_convenio_dpvat,
  vl_consorcio_fundos,
  ds_chave_busca_empresa,
  nm_empresa_consolidada_original,
  nm_grupo_concorrencia_n1,
  nm_grupo_concorrencia_n2,
  ds_tipo_grupo,
  ds_regra_mapeamento_empresa,
  ds_confianca_mapeamento_empresa,
  ds_observacao_mapeamento_empresa,
  fl_alterou_vs_consulta_atual,
  ds_regra_deduplicacao_empresa,
  dt_inicio_vigencia_mapeamento_ramo,
  dt_fim_vigencia_mapeamento_ramo,
  fl_ativo_mapeamento_ramo,
  nr_versao_mapeamento_ramo,
  ds_justificativa_tratamento_ramo,
  ds_fonte_regra_mapeamento_ramo,
  ts_processamento_mapeamento_ramo,
  cd_hash_registro_mapeamento_ramo,
  dt_inicio_vigencia_mapeamento_empresa,
  dt_fim_vigencia_mapeamento_empresa,
  fl_ativo_mapeamento_empresa,
  nr_versao_mapeamento_empresa,
  ds_fonte_regra_mapeamento_empresa,
  ts_processamento_mapeamento_empresa,
  cd_hash_registro_mapeamento_empresa,
  ts_ingestao_bronze,
  nm_arquivo_origem,
  ds_caminho_arquivo_origem,
  ds_pasta_origem,
  id_lote_ingestao,
  nm_tabela_origem,
  nr_ano_mes_referencia_origem,
  cd_hash_linha_bronze,
  ts_processamento_silver,
  cd_hash_registro_silver
)
  WITH s_norm AS (SELECT
      s.*,
      TRY_CAST(
        REGEXP_REPLACE(NULLIF(TRIM(CAST(s.damesano AS STRING)), 'null'), '\.0$', '') AS INT
      ) AS nr_ano_mes_referencia,
      TRY_CAST(
        SUBSTR(
          LPAD(
            REGEXP_REPLACE(NULLIF(TRIM(CAST(s.damesano AS STRING)), 'null'), '\.0$', ''),
            6,
            '0'
          ),
          1,
          4
        ) AS INT
      ) AS nr_ano_referencia,
      TRY_CAST(
        SUBSTR(
          LPAD(
            REGEXP_REPLACE(NULLIF(TRIM(CAST(s.damesano AS STRING)), 'null'), '\.0$', ''),
            6,
            '0'
          ),
          5,
          2
        ) AS INT
      ) AS nr_mes_referencia,
      TO_DATE(
        CONCAT(
          CAST(s.damesano AS STRING),
          '-01'
        )
      ) AS dt_referencia,
      REGEXP_REPLACE(
        NULLIF(TRIM(CAST(s.coenti AS STRING)), 'null'),
        '\.0$',
        ''
      ) AS cd_entidade_susep,
      REGEXP_REPLACE(
        NULLIF(TRIM(CAST(s.cogrupo AS STRING)), 'null'),
        '\.0$',
        ''
      ) AS cd_grupo_empresa_susep_original,
      LPAD(
        REGEXP_REPLACE(NULLIF(TRIM(CAST(s.coramo AS STRING)), 'null'), '\.0$', ''),
        4,
        '0'
      ) AS cd_ramo_susep
    FROM
      susep_bronze.ses_seguros s
  ),
  dados_silver AS (SELECT
      s_norm.nr_ano_mes_referencia,
      s_norm.nr_ano_referencia,
      s_norm.nr_mes_referencia,
      s_norm.dt_referencia,
      s_norm.cd_entidade_susep,
      NULLIF(TRIM(CAST(e.noenti AS STRING)), 'null') AS nm_entidade_susep,
      s_norm.cd_grupo_empresa_susep_original,
      REGEXP_REPLACE(
        NULLIF(TRIM(CAST(e.cogrupo AS STRING)), 'null'),
        '\\.0$',
        ''
      ) AS cd_grupo_empresa_susep_mapeado,
      NULLIF(TRIM(CAST(e.nogrupo AS STRING)), 'null') AS nm_grupo_empresa_susep_mapeado,
      LPAD(
        REGEXP_REPLACE(NULLIF(TRIM(CAST(r.cod_grupo_susep AS STRING)), 'null'), '\\.0$', ''),
        2,
        '0'
      ) AS cd_grupo_ramo_susep,
      NULLIF(TRIM(CAST(r.nome_grupo_susep AS STRING)), 'null') AS nm_grupo_ramo_susep,
      s_norm.cd_ramo_susep,
      NULLIF(TRIM(CAST(r.nome_ramo_susep AS STRING)), 'null') AS nm_ramo_susep,
      LPAD(
        REGEXP_REPLACE(NULLIF(TRIM(CAST(r.cod_ramo_tratado AS STRING)), 'null'), '\\.0$', ''),
        4,
        '0'
      ) AS cd_ramo_tratado,
      NULLIF(TRIM(CAST(r.nome_ramo_tratado AS STRING)), 'null') AS nm_ramo_tratado,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_direto AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_direto AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_direto AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_direto AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.premio_direto AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_direto AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_direto,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_de_seguros AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_de_seguros AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_de_seguros AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_de_seguros AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.premio_de_seguros AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_de_seguros AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_de_seguros,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_retido AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_retido AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_retido AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_retido AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.premio_retido AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_retido AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_retido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_ganho AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_ganho AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_ganho AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_ganho AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.premio_ganho AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_ganho AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_ganho,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistro_direto AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.sinistro_direto AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistro_direto AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistro_direto AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.sinistro_direto AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistro_direto AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_direto,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistro_retido AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.sinistro_retido AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistro_retido AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistro_retido AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.sinistro_retido AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistro_retido AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_retido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.desp_com AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.desp_com AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.desp_com AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.desp_com AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.desp_com AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.desp_com AS STRING)) AS DECIMAL(24, 6))
      END AS vl_despesa_comercializacao,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_emitido2 AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_emitido2 AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_emitido2 AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_emitido2 AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.premio_emitido2 AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_emitido2 AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_emitido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(
            CAST(s_norm.premio_emitido_cap AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_emitido_cap AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_emitido_capitalizacao,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.despesa_resseguros AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(
            CAST(s_norm.despesa_resseguros AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.despesa_resseguros AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.despesa_resseguros AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.despesa_resseguros AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.despesa_resseguros AS STRING)) AS DECIMAL(24, 6))
      END AS vl_despesa_resseguro,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_ocorrido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.receita_resseguro AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.receita_resseguro AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.receita_resseguro AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.receita_resseguro AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.receita_resseguro AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.receita_resseguro AS STRING)) AS DECIMAL(24, 6))
      END AS vl_receita_resseguro,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(
            CAST(s_norm.sinistros_ocorridos_cap AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_ocorrido_capitalizacao,
      CASE
        WHEN
          NULLIF(TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)), 'null') IS NULL
        THEN
          NULL
        WHEN
          TRIM(
            CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        ELSE
          TRY_CAST(
            TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)) AS DECIMAL(24, 6)
          )
      END AS vl_recuperacao_sinistro_ocorrido_capitalizacao,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.rvne AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.rvne AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(REPLACE(TRIM(CAST(s_norm.rvne AS STRING)), '.', ''), ',', '.') AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.rvne AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.rvne AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.rvne AS STRING)) AS DECIMAL(24, 6))
      END AS vl_rvne,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.conveniodpvat AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.conveniodpvat AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.conveniodpvat AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.conveniodpvat AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.conveniodpvat AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.conveniodpvat AS STRING)) AS DECIMAL(24, 6))
      END AS vl_convenio_dpvat,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.consorciosefundos AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.consorciosefundos AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.consorciosefundos AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.consorciosefundos AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.consorciosefundos AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.consorciosefundos AS STRING)) AS DECIMAL(24, 6))
      END AS vl_consorcio_fundos,
      NULLIF(TRIM(CAST(e.search_string AS STRING)), 'null') AS ds_chave_busca_empresa,
      NULLIF(
        TRIM(CAST(e.nome_empresa_consolidada_query_original AS STRING)),
        'null'
      ) AS nm_empresa_consolidada_original,
      NULLIF(TRIM(CAST(e.grupo_concorrencia_n1 AS STRING)), 'null') AS nm_grupo_concorrencia_n1,
      NULLIF(TRIM(CAST(e.grupo_concorrencia_n2 AS STRING)), 'null') AS nm_grupo_concorrencia_n2,
      NULLIF(TRIM(CAST(e.tipo_grupo AS STRING)), 'null') AS ds_tipo_grupo,
      NULLIF(TRIM(CAST(e.regra_aplicada AS STRING)), 'null') AS ds_regra_mapeamento_empresa,
      NULLIF(
        TRIM(CAST(e.confianca_mapeamento AS STRING)),
        'null'
      ) AS ds_confianca_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.observacao AS STRING)), 'null') AS ds_observacao_mapeamento_empresa,
      CASE
        WHEN
          LOWER(TRIM(CAST(e.alterou_vs_query_atual AS STRING))) IN ('true', '1', 'sim', 's', 'yes')
        THEN
          TRUE
        WHEN
          LOWER(TRIM(CAST(e.alterou_vs_query_atual AS STRING))) IN (
            'false', '0', 'não', 'nao', 'n', 'no'
          )
        THEN
          FALSE
        ELSE NULL
      END AS fl_alterou_vs_consulta_atual,
      NULLIF(TRIM(CAST(e.dedup_rule_applied AS STRING)), 'null') AS ds_regra_deduplicacao_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(r.vigencia_inicio AS STRING)), 'null') AS DATE
      ) AS dt_inicio_vigencia_mapeamento_ramo,
      TRY_CAST(
        NULLIF(TRIM(CAST(r.vigencia_fim AS STRING)), 'null') AS DATE
      ) AS dt_fim_vigencia_mapeamento_ramo,
      CASE
        WHEN LOWER(TRIM(CAST(r.flag_ativo AS STRING))) IN ('true', '1', 'sim', 's', 'yes') THEN TRUE
        WHEN
          LOWER(TRIM(CAST(r.flag_ativo AS STRING))) IN ('false', '0', 'não', 'nao', 'n', 'no')
        THEN
          FALSE
        ELSE NULL
      END AS fl_ativo_mapeamento_ramo,
      NULLIF(TRIM(CAST(r.versao_mapeamento AS STRING)), 'null') AS nr_versao_mapeamento_ramo,
      NULLIF(
        TRIM(CAST(r.justificativa_tratamento AS STRING)),
        'null'
      ) AS ds_justificativa_tratamento_ramo,
      NULLIF(TRIM(CAST(r.fonte_regra AS STRING)), 'null') AS ds_fonte_regra_mapeamento_ramo,
      TRY_CAST(
        NULLIF(TRIM(CAST(r.dt_processamento AS STRING)), 'null') AS TIMESTAMP
      ) AS ts_processamento_mapeamento_ramo,
      NULLIF(TRIM(CAST(r.hash_registro AS STRING)), 'null') AS cd_hash_registro_mapeamento_ramo,
      TRY_CAST(
        NULLIF(TRIM(CAST(e.vigencia_inicio AS STRING)), 'null') AS DATE
      ) AS dt_inicio_vigencia_mapeamento_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(e.vigencia_fim AS STRING)), 'null') AS DATE
      ) AS dt_fim_vigencia_mapeamento_empresa,
      CASE
        WHEN LOWER(TRIM(CAST(e.flag_ativo AS STRING))) IN ('true', '1', 'sim', 's', 'yes') THEN TRUE
        WHEN
          LOWER(TRIM(CAST(e.flag_ativo AS STRING))) IN ('false', '0', 'não', 'nao', 'n', 'no')
        THEN
          FALSE
        ELSE NULL
      END AS fl_ativo_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.versao_mapeamento AS STRING)), 'null') AS nr_versao_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.fonte_regra AS STRING)), 'null') AS ds_fonte_regra_mapeamento_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(e.dt_processamento AS STRING)), 'null') AS TIMESTAMP
      ) AS ts_processamento_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.hash_registro AS STRING)), 'null') AS cd_hash_registro_mapeamento_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(s_norm.ingestion_timestamp AS STRING)), 'null') AS TIMESTAMP
      ) AS ts_ingestao_bronze,
      NULLIF(TRIM(CAST(s_norm.source_file AS STRING)), 'null') AS nm_arquivo_origem,
      NULLIF(TRIM(CAST(s_norm.source_file_path AS STRING)), 'null') AS ds_caminho_arquivo_origem,
      NULLIF(TRIM(CAST(s_norm.source_folder AS STRING)), 'null') AS ds_pasta_origem,
      NULLIF(TRIM(CAST(s_norm.ingestion_batch_id AS STRING)), 'null') AS id_lote_ingestao,
      NULLIF(TRIM(CAST(s_norm.source_table_name AS STRING)), 'null') AS nm_tabela_origem,
      TRY_CAST(
        REGEXP_REPLACE(
          NULLIF(TRIM(CAST(s_norm.reference_year_month AS STRING)), 'null'),
          '\\.0$',
          ''
        ) AS INT
      ) AS nr_ano_mes_referencia_origem,
      NULLIF(TRIM(CAST(s_norm.row_hash AS STRING)), 'null') AS cd_hash_linha_bronze
    FROM
      s_norm
        LEFT JOIN susep_bronze.aux_de_para_ramo_susep r
          ON s_norm.cd_ramo_susep
            = LPAD(
              REGEXP_REPLACE(NULLIF(TRIM(CAST(r.cod_ramo_susep AS STRING)), 'null'), '\.0$', ''),
              4,
              '0'
            )
          AND s_norm.dt_referencia
            >= COALESCE(
              TRY_CAST(NULLIF(TRIM(CAST(r.vigencia_inicio AS STRING)), 'null') AS DATE),
              DATE '1900-01-01'
            )
          AND s_norm.dt_referencia
            <= COALESCE(
              TRY_CAST(NULLIF(TRIM(CAST(r.vigencia_fim AS STRING)), 'null') AS DATE),
              DATE '9999-12-31'
            )
          AND COALESCE(
            CASE
              WHEN
                LOWER(TRIM(CAST(r.flag_ativo AS STRING))) IN ('true', '1', 'sim', 's', 'yes')
              THEN
                TRUE
              WHEN
                LOWER(TRIM(CAST(r.flag_ativo AS STRING))) IN ('false', '0', 'não', 'nao', 'n', 'no')
              THEN
                FALSE
              ELSE NULL
            END,
            TRUE
          ) = TRUE
        LEFT JOIN susep_bronze.aux_de_para_grupo_empresa e
          ON s_norm.cd_entidade_susep
            = REGEXP_REPLACE(NULLIF(TRIM(CAST(e.coenti AS STRING)), 'null'), '\.0$', '')
          AND s_norm.dt_referencia
            >= COALESCE(
              TRY_CAST(NULLIF(TRIM(CAST(e.vigencia_inicio AS STRING)), 'null') AS DATE),
              DATE '1900-01-01'
            )
          AND s_norm.dt_referencia
            <= COALESCE(
              TRY_CAST(NULLIF(TRIM(CAST(e.vigencia_fim AS STRING)), 'null') AS DATE),
              DATE '9999-12-31'
            )
          AND COALESCE(
            CASE
              WHEN
                LOWER(TRIM(CAST(e.flag_ativo AS STRING))) IN ('true', '1', 'sim', 's', 'yes')
              THEN
                TRUE
              WHEN
                LOWER(TRIM(CAST(e.flag_ativo AS STRING))) IN ('false', '0', 'não', 'nao', 'n', 'no')
              THEN
                FALSE
              ELSE NULL
            END,
            TRUE
          ) = TRUE
  ),
  base_classificacao_mercado AS (SELECT
      d.*,
      COALESCE(d.cd_ramo_tratado, d.cd_ramo_susep) AS cd_ramo_classificacao,
      COALESCE(
        d.cd_grupo_ramo_susep,
        SUBSTR(COALESCE(d.cd_ramo_tratado, d.cd_ramo_susep), 1, 2)
      ) AS cd_grupo_ramo_classificacao,
      UPPER(COALESCE(d.nm_ramo_tratado, d.nm_ramo_susep, '')) AS nm_ramo_classificacao
    FROM
      dados_silver d
  ),
  classificacao_silver AS (SELECT
      d.*,
      CASE
        WHEN d.cd_ramo_classificacao IS NULL THEN 'NAO_CLASSIFICADO'
        WHEN
          d.cd_ramo_classificacao IN ('0912', '1312')
          OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
        THEN
          'ACUMULACAO_VGBL_SOBREVIVENCIA'
        WHEN
          d.cd_ramo_classificacao = '1603'
          OR d.cd_grupo_ramo_classificacao = '22'
          OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
        THEN
          'PREVIDENCIA_EFPC'
        WHEN
          d.cd_grupo_ramo_classificacao = '19'
          OR d.cd_ramo_classificacao IN ('1202', '1901')
          OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
        THEN
          'SAUDE_SUPLEMENTAR'
        WHEN
          d.cd_ramo_classificacao = '0506'
          OR d.nm_ramo_classificacao RLIKE 'DPVAT'
        THEN
          'SEGURO_DPVAT'
        WHEN
          d.cd_grupo_ramo_classificacao IN ('09', '13')
          OR d.cd_ramo_classificacao RLIKE '^(09|13)'
        THEN
          'SEGUROS_PESSOAS_RISCO'
        WHEN
          d.cd_grupo_ramo_classificacao = '16'
          OR d.cd_ramo_classificacao RLIKE '^16'
        THEN
          'MICROSSEGUROS'
        ELSE 'SEGUROS_DANOS_RESPONSABILIDADES'
      END AS ds_categoria_mercado_susep,
      CASE
        WHEN
          (
            CASE
              WHEN d.cd_ramo_classificacao IS NULL THEN 'NAO_CLASSIFICADO'
              WHEN
                d.cd_ramo_classificacao IN ('0912', '1312')
                OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
              THEN
                'ACUMULACAO_VGBL_SOBREVIVENCIA'
              WHEN
                d.cd_ramo_classificacao = '1603'
                OR d.cd_grupo_ramo_classificacao = '22'
                OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
              THEN
                'PREVIDENCIA_EFPC'
              WHEN
                d.cd_grupo_ramo_classificacao = '19'
                OR d.cd_ramo_classificacao IN ('1202', '1901')
                OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
              THEN
                'SAUDE_SUPLEMENTAR'
              WHEN
                d.cd_ramo_classificacao = '0506'
                OR d.nm_ramo_classificacao RLIKE 'DPVAT'
              THEN
                'SEGURO_DPVAT'
              WHEN
                d.cd_grupo_ramo_classificacao IN ('09', '13')
                OR d.cd_ramo_classificacao RLIKE '^(09|13)'
              THEN
                'SEGUROS_PESSOAS_RISCO'
              WHEN
                d.cd_grupo_ramo_classificacao = '16'
                OR d.cd_ramo_classificacao RLIKE '^16'
              THEN
                'MICROSSEGUROS'
              ELSE 'SEGUROS_DANOS_RESPONSABILIDADES'
            END
          ) IN (
            'SEGUROS_DANOS_RESPONSABILIDADES',
            'SEGUROS_PESSOAS_RISCO',
            'MICROSSEGUROS',
            'SEGURO_DPVAT'
          )
        THEN
          TRUE
        ELSE FALSE
      END AS fl_base_analitica_seguros,
      CASE
        WHEN
          d.cd_ramo_classificacao = '0506'
          OR d.nm_ramo_classificacao RLIKE 'DPVAT'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_dpvat,
      CASE
        WHEN
          d.cd_ramo_classificacao IN ('0912', '1312')
          OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_acumulacao,
      CASE
        WHEN
          d.cd_ramo_classificacao = '1603'
          OR d.cd_grupo_ramo_classificacao = '22'
          OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_previdencia,
      CASE
        WHEN
          d.cd_grupo_ramo_classificacao = '19'
          OR d.cd_ramo_classificacao IN ('1202', '1901')
          OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_saude,
      CASE
        WHEN
          d.cd_ramo_classificacao IS NULL
        THEN
          'Não classificado porque não houve código de ramo suficiente para aplicar a regra de mercado.'
        WHEN
          d.cd_ramo_classificacao IN ('0912', '1312')
          OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
        THEN
          'Excluído da visão principal de seguros por regra de VGBL/sobrevivência/acumulação.'
        WHEN
          d.cd_ramo_classificacao = '1603'
          OR d.cd_grupo_ramo_classificacao = '22'
          OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
        THEN
          'Excluído da visão principal de seguros por regra de previdência, EFPC ou microsseguro previdenciário.'
        WHEN
          d.cd_grupo_ramo_classificacao = '19'
          OR d.cd_ramo_classificacao IN ('1202', '1901')
          OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
        THEN
          'Excluído da visão principal de seguros por regra de saúde suplementar ou saúde resseguro.'
        WHEN
          d.cd_ramo_classificacao = '0506'
          OR d.nm_ramo_classificacao RLIKE 'DPVAT'
        THEN
          'Mantido na base de seguros, mas identificado por flag própria para análises com ou sem DPVAT.'
        WHEN
          d.cd_grupo_ramo_classificacao IN ('09', '13')
          OR d.cd_ramo_classificacao RLIKE '^(09|13)'
        THEN
          'Mantido na base de seguros como seguro de pessoas de risco, exceto regras específicas de acumulação já tratadas.'
        WHEN
          d.cd_grupo_ramo_classificacao = '16'
          OR d.cd_ramo_classificacao RLIKE '^16'
        THEN
          'Mantido na base de seguros como microsseguro, exceto microsseguro previdenciário já tratado.'
        ELSE
          'Mantido na base de seguros como seguro de danos, responsabilidades ou demais ramos securitários.'
      END AS ds_criterio_classificacao_mercado
    FROM
      base_classificacao_mercado d
  )
  SELECT
    d.nr_ano_mes_referencia,
    d.nr_ano_referencia,
    d.nr_mes_referencia,
    d.dt_referencia,
    d.cd_entidade_susep,
    d.nm_entidade_susep,
    d.cd_grupo_empresa_susep_original,
    d.cd_grupo_empresa_susep_mapeado,
    d.nm_grupo_empresa_susep_mapeado,
    d.cd_grupo_ramo_susep,
    d.nm_grupo_ramo_susep,
    d.cd_ramo_susep,
    d.nm_ramo_susep,
    d.cd_ramo_tratado,
    d.nm_ramo_tratado,
    d.ds_categoria_mercado_susep,
    d.fl_base_analitica_seguros,
    d.fl_ramo_dpvat,
    d.fl_ramo_acumulacao,
    d.fl_ramo_previdencia,
    d.fl_ramo_saude,
    d.ds_criterio_classificacao_mercado,
    d.vl_premio_direto,
    d.vl_premio_de_seguros,
    d.vl_premio_retido,
    d.vl_premio_ganho,
    d.vl_sinistro_direto,
    d.vl_sinistro_retido,
    d.vl_despesa_comercializacao,
    d.vl_premio_emitido,
    d.vl_premio_emitido_capitalizacao,
    d.vl_despesa_resseguro,
    d.vl_sinistro_ocorrido,
    d.vl_receita_resseguro,
    d.vl_sinistro_ocorrido_capitalizacao,
    d.vl_recuperacao_sinistro_ocorrido_capitalizacao,
    d.vl_rvne,
    d.vl_convenio_dpvat,
    d.vl_consorcio_fundos,
    d.ds_chave_busca_empresa,
    d.nm_empresa_consolidada_original,
    d.nm_grupo_concorrencia_n1,
    d.nm_grupo_concorrencia_n2,
    d.ds_tipo_grupo,
    d.ds_regra_mapeamento_empresa,
    d.ds_confianca_mapeamento_empresa,
    d.ds_observacao_mapeamento_empresa,
    d.fl_alterou_vs_consulta_atual,
    d.ds_regra_deduplicacao_empresa,
    d.dt_inicio_vigencia_mapeamento_ramo,
    d.dt_fim_vigencia_mapeamento_ramo,
    d.fl_ativo_mapeamento_ramo,
    d.nr_versao_mapeamento_ramo,
    d.ds_justificativa_tratamento_ramo,
    d.ds_fonte_regra_mapeamento_ramo,
    d.ts_processamento_mapeamento_ramo,
    d.cd_hash_registro_mapeamento_ramo,
    d.dt_inicio_vigencia_mapeamento_empresa,
    d.dt_fim_vigencia_mapeamento_empresa,
    d.fl_ativo_mapeamento_empresa,
    d.nr_versao_mapeamento_empresa,
    d.ds_fonte_regra_mapeamento_empresa,
    d.ts_processamento_mapeamento_empresa,
    d.cd_hash_registro_mapeamento_empresa,
    d.ts_ingestao_bronze,
    d.nm_arquivo_origem,
    d.ds_caminho_arquivo_origem,
    d.ds_pasta_origem,
    d.id_lote_ingestao,
    d.nm_tabela_origem,
    d.nr_ano_mes_referencia_origem,
    d.cd_hash_linha_bronze,
    CURRENT_TIMESTAMP() AS ts_processamento_silver,
    sha2(
      concat_ws(
        '||',
        COALESCE(CAST(d.nr_ano_mes_referencia AS STRING), ''),
        COALESCE(CAST(d.nr_ano_referencia AS STRING), ''),
        COALESCE(CAST(d.nr_mes_referencia AS STRING), ''),
        COALESCE(CAST(d.dt_referencia AS STRING), ''),
        COALESCE(CAST(d.cd_entidade_susep AS STRING), ''),
        COALESCE(CAST(d.nm_entidade_susep AS STRING), ''),
        COALESCE(CAST(d.cd_grupo_empresa_susep_original AS STRING), ''),
        COALESCE(CAST(d.cd_grupo_empresa_susep_mapeado AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_empresa_susep_mapeado AS STRING), ''),
        COALESCE(CAST(d.cd_grupo_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.cd_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.nm_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.cd_ramo_tratado AS STRING), ''),
        COALESCE(CAST(d.nm_ramo_tratado AS STRING), ''),
        COALESCE(CAST(d.ds_categoria_mercado_susep AS STRING), ''),
        COALESCE(CAST(d.fl_base_analitica_seguros AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_dpvat AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_acumulacao AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_previdencia AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_saude AS STRING), ''),
        COALESCE(CAST(d.ds_criterio_classificacao_mercado AS STRING), ''),
        COALESCE(CAST(d.vl_premio_direto AS STRING), ''),
        COALESCE(CAST(d.vl_premio_de_seguros AS STRING), ''),
        COALESCE(CAST(d.vl_premio_retido AS STRING), ''),
        COALESCE(CAST(d.vl_premio_ganho AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_direto AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_retido AS STRING), ''),
        COALESCE(CAST(d.vl_despesa_comercializacao AS STRING), ''),
        COALESCE(CAST(d.vl_premio_emitido AS STRING), ''),
        COALESCE(CAST(d.vl_premio_emitido_capitalizacao AS STRING), ''),
        COALESCE(CAST(d.vl_despesa_resseguro AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_ocorrido AS STRING), ''),
        COALESCE(CAST(d.vl_receita_resseguro AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_ocorrido_capitalizacao AS STRING), ''),
        COALESCE(CAST(d.vl_recuperacao_sinistro_ocorrido_capitalizacao AS STRING), ''),
        COALESCE(CAST(d.vl_rvne AS STRING), ''),
        COALESCE(CAST(d.vl_convenio_dpvat AS STRING), ''),
        COALESCE(CAST(d.vl_consorcio_fundos AS STRING), ''),
        COALESCE(CAST(d.ds_chave_busca_empresa AS STRING), ''),
        COALESCE(CAST(d.nm_empresa_consolidada_original AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_concorrencia_n1 AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_concorrencia_n2 AS STRING), ''),
        COALESCE(CAST(d.ds_tipo_grupo AS STRING), ''),
        COALESCE(CAST(d.ds_regra_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ds_confianca_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ds_observacao_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.fl_alterou_vs_consulta_atual AS STRING), ''),
        COALESCE(CAST(d.ds_regra_deduplicacao_empresa AS STRING), ''),
        COALESCE(CAST(d.dt_inicio_vigencia_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.dt_fim_vigencia_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.fl_ativo_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.nr_versao_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.ds_justificativa_tratamento_ramo AS STRING), ''),
        COALESCE(CAST(d.ds_fonte_regra_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.ts_processamento_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.cd_hash_registro_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.dt_inicio_vigencia_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.dt_fim_vigencia_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.fl_ativo_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.nr_versao_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ds_fonte_regra_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ts_processamento_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.cd_hash_registro_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ts_ingestao_bronze AS STRING), ''),
        COALESCE(CAST(d.nm_arquivo_origem AS STRING), ''),
        COALESCE(CAST(d.ds_caminho_arquivo_origem AS STRING), ''),
        COALESCE(CAST(d.ds_pasta_origem AS STRING), ''),
        COALESCE(CAST(d.id_lote_ingestao AS STRING), ''),
        COALESCE(CAST(d.nm_tabela_origem AS STRING), ''),
        COALESCE(CAST(d.nr_ano_mes_referencia_origem AS STRING), ''),
        COALESCE(CAST(d.cd_hash_linha_bronze AS STRING), '')
      ),
      256
    ) AS cd_hash_registro_silver
  FROM
    classificacao_silver d;